In [1]:
#importing all the necessary libraries

import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 10)

In [2]:
#importing the data
customers = pd.read_csv('olist_customers_dataset.csv')
geolocation = pd.read_csv('olist_geolocation_dataset.csv')
order_items = pd.read_csv('olist_order_items_dataset.csv')
payments = pd.read_csv('olist_order_payments_dataset.csv')
reviews = pd.read_csv("olist_order_reviews_dataset.csv")
orders = pd.read_csv("olist_orders_dataset.csv")
products = pd.read_csv("olist_products_dataset.csv")
sellers = pd.read_csv("olist_sellers_dataset.csv")
category_translation = pd.read_csv("product_category_name_translation.csv")

In [3]:
## dataset overview

#   dataset    | Number of rows | Number of columns |       Primary Key       |       Foreign Key       | Missing Values |       Business Purpose
#  customers        99441                 5                 customer_id                   No           0                Customer Information
# geolocation      1000163                5                No primary key             No primary key            0                    Geolocation
# order items      112650                 7            order_id + order_item_id          Product_id, Order_id         0                 Order Items Detail
#  payments        103886                 5          Order_id, Payment_Sequntial        Order_id                0                 Payment Details
#  reviews         99224                  7          order_id, review_id              Order_id      [review_comment_title 87656,   Customer Reviews
#                                                                                                   review_comment_message	58247]
#  orders          99441                  8                order_id                   customer_id   [order_approved_at	160,           Order Details
#                                                                                                    order_delivered_carrier_date	1783,
#                                                                                                     order_delivered_customer_date	2965]   Product Details
#  products        32951                  9                 product_id            product_category_name [product_category_name	610,
#                                                                                                         product_name_lenght	610,
#                                                                                                       product_description_lenght	610,
#                                                                                                             product_photos_qty	610,
#                                                                                                               product_weight_g	2,
#                                                                                                             product_length_cm	2,
#                                                                                                             product_height_cm	2,
#                                                                                                             product_width_cm	2]
# sellers           3095                  4                 seller_id                 No Foreign Key            0                     Seller Information
# (category_
# translation)     71                     2         product_category_name_english     No Foreign Key            0                  Product category name translation

In [4]:
# Merge Strategy

#| Step | Left Table | Right Table | Join Key | Relationship | Join Type | Expected Result |
#|------|------------|-------------|----------|--------------|-----------|-----------------|
#| 1 | Order Items | Orders | order_id | Many : One | Left | Add order information |
#| 2 | Result | Customers | customer_id | Many : One | Left | Add customer details |
#| 3 | Result | Products | product_id | Many : One | Left | Add product information |
#| 4 | Result | Category Translation | product_category_name | Many : One | Left | Translate category names |
#| 5 | Result | Sellers | seller_id | Many : One | Left | Add seller information |

In [5]:
#merge order items with orders

master_df = pd.merge(order_items, orders, how = "left", on = "order_id", validate = "many_to_one")  #validate is life safety net to check the connection, is it many to one or not

In [6]:
#this is for testing purpose for the indicator parameter
master_df2 = pd.merge(
    order_items,
    orders,
    how="left",
    on="order_id",
    indicator=True
)
master_df2["_merge"].value_counts()

,count
_merge,
both,112650
left_only,0
right_only,0


In [7]:
### Merge Validation

#The merge between **Order Items** and **Orders** was successful.

#- All 112,650 order item records were preserved.
#- Every order item matched a valid order.
#- No orphan transactions were identified.
#- The dataset now contains both product-level and order-level information.

#The analytical grain remains **one row per order item**, which is suitable for downstream revenue, product, and seller analysis.

In [8]:
#merging master_df with customers
master_dfcustomers = pd.merge(master_df, customers, how = "left", on = "customer_id", validate = "many_to_one")

In [9]:
## Merge 2: Orders → Customers

### Objective

#Enrich the transactional dataset with customer-level information.

### Relationship

#- Left Table: Master Dataset (Order Items + Orders)
#- Right Table: Customers
#- Join Key: customer_id
#- Relationship: Many-to-One
#- Join Type: Left Join

### Validation

#- Row count remained unchanged (112,650).
#- All records matched successfully.
#- No missing customer identifiers were introduced.
#- Relationship validation passed using `validate="many_to_one"`.

### Conclusion

#The master dataset now contains transaction-level and customer-level information while preserving the analytical grain of one row per order item.

In [10]:
#merging masterdf_customer to product

master_dfproducts = pd.merge(master_dfcustomers, products, how = "left", on = "product_id", validate = "many_to_one", indicator = True)

In [11]:
print(f"Rows before merge : {master_dfcustomers.shape[0]}")
print(f"Rows after merge : {master_dfproducts.shape[0]}")

Rows before merge : 112650
Rows after merge : 112650


In [12]:
master_dfproducts["_merge"].value_counts()

,count
_merge,
both,112650
left_only,0
right_only,0


In [13]:
master_dfproducts.isnull().sum().sort_values(ascending = False)

,0
order_delivered_customer_date,2454
product_category_name,1603
product_description_lenght,1603
product_name_lenght,1603
product_photos_qty,1603
...,...
customer_unique_id,0
order_estimated_delivery_date,0
customer_city,0
customer_zip_code_prefix,0


In [16]:
#merging master_dfproducts to category_translation

master_dfcategory = pd.merge(master_dfproducts, category_translation, how = "left", on = "product_category_name", validate = "many_to_one", indicator = True)

In [17]:
print(f" Rows before merge : {len(master_dfproducts)}")
print(f" Rows after merge : {len(master_dfcategory)}")

 Rows before merge : 112650
 Rows after merge : 112650


In [18]:
master_dfcategory["_merge"].value_counts()

,count
_merge,
both,111023
left_only,1627
right_only,0


In [19]:
master_dfcategory.loc[master_dfcategory["_merge"] == "left_only","product_category_name"].value_counts(dropna = False)

,count
product_category_name,
NaN,1603
portateis_cozinha_e_preparadores_de_alimentos,15
pc_gamer,9


In [20]:
master_dfcategory[
    [
        "product_category_name",
        "product_category_name_english"
    ]
].isnull().sum()

,0
product_category_name,1603
product_category_name_english,1627


In [21]:
#filling missing categories

master_dfcategory["product_category_name"].fillna("Unknown", inplace = True)

/tmp/ipykernel_4202/3565974951.py:3: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  master_dfcategory["product_category_name"].fillna("Unknown", inplace = True)


In [22]:
#checking anomalies
products[
    products["product_category_name"].isin([
        "pc_gamer",
        "portateis_cozinha_e_preparadores_de_alimentos"
    ])
]

,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
1628,0105b5323d24fc655f73052694dbbb3a,pc_gamer,59.0,621.0,4.0,2839.0,19.0,16.0,18.0
5821,6fd83eb3e0799b775e4f946bd66657c0,portateis_cozinha_e_preparadores_de_alimentos,52.0,280.0,1.0,1200.0,25.0,33.0,25.0
7325,5d923ead886c44b86845f69e50520c3e,portateis_cozinha_e_preparadores_de_alimentos,58.0,284.0,1.0,1200.0,25.0,33.0,25.0
7478,6727051471a0fc4a0e7737b57bff2549,pc_gamer,60.0,1532.0,3.0,650.0,16.0,22.0,20.0
8819,bed164d9d628cf0593003389c535c6e0,portateis_cozinha_e_preparadores_de_alimentos,54.0,382.0,2.0,850.0,30.0,21.0,22.0
...,...,...,...,...,...,...,...,...,...
16930,dbe520fb381ad695a7e1f2807d20c765,pc_gamer,60.0,840.0,6.0,800.0,18.0,22.0,22.0
17800,c7a3f1a7f9eef146cc499368b578b884,portateis_cozinha_e_preparadores_de_alimentos,52.0,1372.0,5.0,7350.0,40.0,30.0,23.0
18610,7afdd65f79f63819ff5bee328843fa37,portateis_cozinha_e_preparadores_de_alimentos,48.0,305.0,1.0,750.0,20.0,20.0,20.0
26890,a4756663d007b0cd1af865754d08d968,portateis_cozinha_e_preparadores_de_alimentos,60.0,1304.0,4.0,650.0,22.0,6.0,14.0


In [23]:
#filling values with manual categories

manual_category_mapping = {
    "pc_gamer": "PC Gamer",
    "portateis_cozinha_e_preparadores_de_alimentos": "Portable Kitchen Appliances"
}

master_dfcategory["product_category_name_english"] = (
    master_dfcategory["product_category_name_english"]
    .fillna(
        master_dfcategory["product_category_name"].map(manual_category_mapping)
    )
)

In [24]:
### Data Quality Observation – Category Translation

#During the merge between the Products table and the Category Translation lookup table, 1,627 transaction records did not find a matching English category.

#Further investigation showed two distinct causes:

#1. **1,603 transaction records** belong to products with a missing `product_category_name` in the source Products table.

#2. **24 transaction records** belong to two valid Portuguese product categories:
#   - `pc_gamer`
#   - `portateis_cozinha_e_preparadores_de_alimentos`

#These categories exist in the Products table but are absent from the Category Translation lookup table, indicating an incomplete reference dataset rather than an error in the transactional data.

In [26]:
#removing previous _merge columns
master_dfcategory.drop(columns = "_merge", inplace = True)

In [27]:
#merging seller with master_dfcategory

master_final = pd.merge(master_dfcategory, sellers, how = "left", on = "seller_id", validate = "many_to_one", indicator = True)

In [29]:
print(f"rows before merge : {len(master_dfcategory)}")
print(f"rows after merge : {len(master_final)}")

rows before merge : 112650
rows after merge : 112650


In [31]:
master_final["_merge"].value_counts()

,count
_merge,
both,112650
left_only,0
right_only,0


In [34]:
master_final.drop(columns = "_merge", inplace = True)

In [32]:
#final QA check
master_final.shape

(112650, 31)

In [35]:
master_final.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 112650 entries, 0 to 112649
Data columns (total 30 columns):
 #   Column                         Non-Null Count   Dtype  
---  ------                         --------------   -----  
 0   order_id                       112650 non-null  object 
 1   order_item_id                  112650 non-null  int64  
 2   product_id                     112650 non-null  object 
 3   seller_id                      112650 non-null  object 
 4   shipping_limit_date            112650 non-null  object 
 5   price                          112650 non-null  float64
 6   freight_value                  112650 non-null  float64
 7   customer_id                    112650 non-null  object 
 8   order_status                   112650 non-null  object 
 9   order_purchase_timestamp       112650 non-null  object 
 10  order_approved_at              112635 non-null  object 
 11  order_delivered_carrier_date   111456 non-null  object 
 12  order_delivered_customer_date 

In [36]:
master_final.isnull().sum().sort_values(ascending = False)

,0
order_delivered_customer_date,2454
product_description_lenght,1603
product_category_name_english,1603
product_name_lenght,1603
product_photos_qty,1603
...,...
customer_state,0
customer_unique_id,0
seller_zip_code_prefix,0
seller_city,0


In [39]:
#missing percentage
(master_final.isnull().mean() * 100).sort_values(ascending = False)

,0
order_delivered_customer_date,2.178429
product_description_lenght,1.422992
product_category_name_english,1.422992
product_name_lenght,1.422992
product_photos_qty,1.422992
...,...
customer_state,0.000000
customer_unique_id,0.000000
seller_zip_code_prefix,0.000000
seller_city,0.000000


In [40]:
#checking duplicate
master_final.duplicated().sum()

np.int64(0)

In [43]:
#memory usage in MB
master_final.memory_usage(deep = True).sum() / 1024**2

np.float64(140.91435527801514)

In [44]:
#descriptive statistics
master_final.describe()

,order_item_id,price,freight_value,customer_zip_code_prefix,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm,seller_zip_code_prefix
count,112650.000000,112650.000000,112650.000000,112650.000000,111047.000000,111047.000000,111047.000000,112632.000000,112632.000000,112632.000000,112632.000000,112650.000000
mean,1.197834,120.653739,19.990320,35119.309090,48.775978,787.867029,2.209713,2093.672047,30.153669,16.593766,22.996546,24439.170431
std,0.705124,183.633928,15.806405,29866.120801,10.025581,652.135608,1.721438,3751.596884,16.153449,13.443483,11.707268,27596.030909
min,1.000000,0.850000,0.000000,1003.000000,5.000000,4.000000,1.000000,0.000000,7.000000,2.000000,6.000000,1001.000000
25%,1.000000,39.900000,13.080000,11310.000000,42.000000,348.000000,1.000000,300.000000,18.000000,8.000000,15.000000,6429.000000
50%,1.000000,74.990000,16.260000,24340.000000,52.000000,603.000000,1.000000,700.000000,25.000000,13.000000,20.000000,13568.000000
75%,1.000000,134.900000,21.150000,59028.750000,57.000000,987.000000,3.000000,1800.000000,38.000000,20.000000,30.000000,27930.000000
max,21.000000,6735.000000,409.680000,99990.000000,76.000000,3992.000000,20.000000,40425.000000,105.000000,105.000000,118.000000,99730.000000


In [49]:
#dataset overview

#master dataset
print(f"Rows    : {master_final.shape[0]:,}")
print(f"Columns : {master_final.shape[1]}")

Rows    : 112,650
Columns : 30


In [50]:
### Observation

#The final analytical dataset contains one row per order item while preserving customer, order, seller, and product information.

#This dataset will serve as the primary analytical table for exploratory data analysis and business intelligence.

In [53]:
#missing report
missing_report = (
    pd.DataFrame({
        "Missing Count": master_final.isnull().sum(),
        "Missing %": (
            master_final.isnull().mean() * 100
        ).round(2)
    })
    .sort_values("Missing Count", ascending=False)
)

missing_report

,Missing Count,Missing %
order_delivered_customer_date,2454,2.18
product_description_lenght,1603,1.42
product_category_name_english,1603,1.42
product_name_lenght,1603,1.42
product_photos_qty,1603,1.42
...,...,...
product_width_cm,18,0.02
product_weight_g,18,0.02
product_height_cm,18,0.02
product_length_cm,18,0.02


#Feature Engineering

In [56]:
#converting to datetime format to date columns

date_columns = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

for col in date_columns:
  master_final[col] = pd.to_datetime(master_final[col])

In [57]:
#finding delivery days

master_final["delivery_days"] = (master_final["order_delivered_customer_date"] - master_final["order_purchase_timestamp"]).dt.days

In [58]:
#approval efficiency

master_final["approval_efficiency"] = (master_final["order_approved_at"] - master_final["order_purchase_timestamp"]).dt.total_seconds()/3600

In [59]:
#purchase month

master_final["purchase_month"] = master_final["order_purchase_timestamp"].dt.month_name()

In [60]:
#purchase year

master_final["purchase_year"] = master_final["order_purchase_timestamp"].dt.year

In [61]:
#purchase weekdays

master_final["purchase_weekdays"] = master_final["order_purchase_timestamp"].dt.day_name()

In [62]:
#delivery status

master_final["Is_Delivered"] = master_final["order_status"] == "delivered"

In [69]:
#freight ration

master_final["freight_ratio"] = ((master_final["freight_value"] / master_final["price"]).round(2)) *100

In [64]:
#item order value

master_final["item_order_value"] = master_final["price"] + master_final["freight_value"]

In [71]:
master_final.head(5)

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm,product_category_name_english,seller_zip_code_prefix,seller_city,seller_state,delivery_days,approval_efficiency,purchase_month,purchase_year,purchase_weekdays,Is_Delivered,freight_ratio,item_order_value
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29,3ce436f183e68e07877b285a838db11a,delivered,2017-09-13 08:59:02,2017-09-13 09:45:35,2017-09-19 18:34:16,2017-09-20 23:43:48,2017-09-29,871766c5855e863f6eccc05f988b23cb,28013,campos dos goytacazes,RJ,cool_stuff,58.0,598.0,4.0,650.0,28.0,9.0,14.0,cool_stuff,27277,volta redonda,SP,7.0,0.775833,September,2017,Wednesday,True,23.0,72.19
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93,f6dd3ec061db4e3987629fe6b26e5cce,delivered,2017-04-26 10:53:06,2017-04-26 11:05:13,2017-05-04 14:35:00,2017-05-12 16:04:24,2017-05-15,eb28e67c4c0b83846050ddfb8a35d051,15775,santa fe do sul,SP,pet_shop,56.0,239.0,2.0,30000.0,50.0,30.0,40.0,pet_shop,3471,sao paulo,SP,16.0,0.201944,April,2017,Wednesday,True,8.0,259.83
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87,6489ae5e4333f3693df5ad4372dab6d3,delivered,2018-01-14 14:33:31,2018-01-14 14:48:30,2018-01-16 12:36:48,2018-01-22 13:19:16,2018-02-05,3818d81c6709e39d06b2738a8d3a2474,35661,para de minas,MG,moveis_decoracao,59.0,695.0,2.0,3050.0,33.0,13.0,33.0,furniture_decor,37564,borda da mata,MG,7.0,0.249722,January,2018,Sunday,True,9.0,216.87
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.99,12.79,d4eb9395c8c0431ee92fce09860c5a06,delivered,2018-08-08 10:00:35,2018-08-08 10:10:18,2018-08-10 13:28:00,2018-08-14 13:32:39,2018-08-20,af861d436cfc08b2c2ddefd0ba074622,12952,atibaia,SP,perfumaria,42.0,480.0,1.0,200.0,16.0,10.0,15.0,perfumery,14403,franca,SP,6.0,0.161944,August,2018,Wednesday,True,98.0,25.78
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.90,18.14,58dbd0b2d70206bf40e62cd34e84d795,delivered,2017-02-04 13:57:51,2017-02-04 14:10:13,2017-02-16 09:46:09,2017-03-01 16:42:31,2017-03-17,64b576fb70d441e8f1b2d7d446e483c5,13226,varzea paulista,SP,ferramentas_jardim,59.0,409.0,1.0,3750.0,35.0,40.0,30.0,garden_tools,87900,loanda,PR,25.0,0.206111,February,2017,Saturday,True,9.0,218.04
